# Step 3: LightGBM engagement classifier (our base model)

**Pipeline.** `step1_data_preprocess.ipynb` → `step2_baselines.ipynb` → `step3_lightgbm.ipynb` (this notebook).

Our base model, as in the interim report: a **binary classifier** predicts the probability that the user engages (click, cart or purchase) with each displayed product, and each slate is ranked by that probability. It is trained with binary cross-entropy (BCE) on step 1's features. This notebook compares it with **every baseline step 2 saved** (simple rules, MF and team 3's model), on the same users and full slates, scored by the same metric code. Two extensions are compared against this model on validation: LightGBM LambdaRank (`step3_lambdarank.ipynb`: same binary labels, listwise loss) and the attention hybrid (`step3_attention.ipynb`).

| # | Decision | Choice |
|---|---|---|
| D17 | Model | LightGBM, `objective="binary"` (BCE) |
| D18 | Training rows | Step 1's `fit` rows: every positive plus 20 random negatives per train slate (D8) |
| D19 | Early stopping | Stop when validation NDCG@10 (LightGBM's own, binary relevance) has not improved for 100 rounds. NDCG is only the stopping rule; the loss is BCE. Slates were shuffled in step 1, so LightGBM's tie handling does not see the stored order |
| D20 | Hyperparameters | Hand-set values in `PARAMS` (Section 2). A random search on validation (12 of 15 planned configurations ran within the 45-minute limit) found no proven gain: the best trial scored Hit@10 +0.0020 (95% CI −0.0003 to +0.0044) and NDCG@10 −0.0004 (CI −0.0015 to +0.0009) against these values, so they are kept. The search is now off (`RUN_TUNING = False`): Section 2 prints the saved trial table and retrains the top trial to re-check its gain. Tuning never uses test |
| D21 | Reporting | Train (10,000 full slates, D12) and validation for our model **and every step 2 baseline**; **test once, at the end** (`RUN_TEST = True`). Gain over the best baseline with a paired bootstrap CI. Every run uploads to its own new `RESULT_FOLDER`, never to `BT4222_artifacts` |

## How to run in Colab
1. Sign in with an account that can see the shared `BT4222_artifacts` folder (read by its fixed id; nothing is copied into your My Drive).
2. Runtime → Run all (CPU is enough). Keep all outputs when you save.
3. `LOAD_MODEL_FROM` names the frozen run's results folder: the notebook loads `lgbm_bce.txt` from it instead of retraining, so the final test run scores exactly the model chosen on validation. Whoever runs it needs that folder shared with them. Set it to `""` to train a new model.
4. The last cell uploads the results to a new folder named in `RESULT_FOLDER` (it includes the date and time, so runs never overwrite each other). Post that folder name in the group chat.

## 1. Setup

In [ ]:
import os, sys, json, time, gc
import numpy as np
import pandas as pd
import lightgbm as lgb

IN_COLAB = "google.colab" in sys.modules

# ---- config: change these, nothing else ----
ART_DIR = os.environ.get("JD_ART_DIR", "/content/artifacts" if IN_COLAB else ".")    # step 1 and step 2 files
OUT_DIR = os.environ.get("JD_OUT_DIR", "/content/results" if IN_COLAB else "results") # result tables and the model
DRIVE_ARTIFACT_FOLDER = "BT4222_artifacts"     # shared folder with step 1 and 2 files (read by fixed id, see _folder_id)
SEED = 42
NDCG_GAIN = "linear"                           # must match step 2: "linear" (gain = label) or "exp" (2^label - 1)
RUN_TEST = True                               # keep False until the final run: test is scored once, at the end
RUN_TUNING = False                             # the search ran on 4 Nov (D20); False prints its saved trials instead
TUNING_FOLDER = "BT4222_step3_tuning_search"   # Drive folder that the 4 Nov search uploaded tuning_trials.csv to
# new Drive folder for this run's results: never BT4222_artifacts, never reused (date and time in the name)
RESULT_FOLDER = f"BT4222_step3_lightgbm_{'test' if RUN_TEST else 'val'}_" + pd.Timestamp.now(tz="Asia/Singapore").strftime("%d%b_%H%M")
EXPECTED_VAL = {"hit@10": 0.7188, "ndcg@10": 0.4146}   # validation figures the model was frozen with; a re-run must match
LOAD_MODEL_FROM = "BT4222_step3_lightgbm_val_08Nov_1257"   # frozen run (8 Nov): load its saved model instead of retraining. "" = train a new model
GROUP_FOLDER_ID = "1YpCRQtsc8dqPi6Q4E9yTfc0wOkpCJ3Z8"     # BT4222-Group-15, where the frozen run folders are
# --------------------------------------------
assert RESULT_FOLDER != DRIVE_ARTIFACT_FOLDER, "never write to the shared artifacts folder"
os.makedirs(OUT_DIR, exist_ok=True)
print("results will upload to My Drive/" + RESULT_FOLDER)

def _drive():
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    return build("drive", "v3")

def _folder_id(svc):
    return "1Yb4AGglIJV4Pdd8Sx6NozG8Klh6uweqH"

def drive_download(names):
    # fetch the named files from My Drive/DRIVE_ARTIFACT_FOLDER into ART_DIR, skipping ones already there
    import io
    from googleapiclient.http import MediaIoBaseDownload
    if not names:
        return
    svc = _drive(); fid = _folder_id(svc); os.makedirs(ART_DIR, exist_ok=True)
    for f in names:
        found = svc.files().list(q=f"'{fid}' in parents and name='{f}' and trashed=false", fields="files(id)").execute()["files"]
        if not found:
            raise FileNotFoundError(f"{f} is not in My Drive/{DRIVE_ARTIFACT_FOLDER}. Run the step that makes it first.")
        with io.FileIO(f"{ART_DIR}/{f}", "wb") as fh:
            dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=found[0]["id"]), chunksize=100 * 1024 * 1024)
            done = False
            while not done:
                _, done = dl.next_chunk()
        print("downloaded", f)

def load_frozen_model(fname):
    """Load the frozen model file fname from the Drive folder LOAD_MODEL_FROM (yours, or shared with you)."""
    path = f"{OUT_DIR}/{fname}"
    if not os.path.exists(path):
        import io
        from googleapiclient.http import MediaIoBaseDownload
        svc = _drive()
        q = f"name='{LOAD_MODEL_FROM}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        # look inside the group folder first, by its id: a name search misses folders you reach only through the group folder
        folders = (svc.files().list(q=f"'{GROUP_FOLDER_ID}' in parents and {q}", fields="files(id)").execute()["files"]
                   or svc.files().list(q=q, fields="files(id)").execute()["files"])
        hits = []
        for fo in folders:
            hits += svc.files().list(q=f"'{fo['id']}' in parents and name='{fname}' and trashed=false",
                                     fields="files(id)").execute()["files"]
        if len(hits) != 1:
            raise FileNotFoundError(f"expected one {fname} in a Drive folder named {LOAD_MODEL_FROM}, found {len(hits)}. "
                                    "Ask the person who ran it to share that folder with you.")
        with io.FileIO(path, "wb") as fh:
            dl = MediaIoBaseDownload(fh, svc.files().get_media(fileId=hits[0]["id"])); done = False
            while not done:
                _, done = dl.next_chunk()
    m = lgb.Booster(model_file=path)
    m.best_iteration = m.current_iteration()   # the saved file holds only the trees up to the frozen run's best iteration
    print(f"loaded frozen model {fname} from My Drive/{LOAD_MODEL_FROM}: {m.current_iteration()} trees, no retraining")
    return m

# Shared metric definitions (src/metrics.py). On Colab without the repo, an identical copy is defined here.
try:
    from src.metrics import MetricAccumulator, bootstrap_ci
except ImportError:
    def _rank(scores, rng):
        # descending score, ties broken uniformly at random (never by stored order: the raw file stores engaged items first)
        return np.lexsort((rng.random(len(scores)), -np.asarray(scores, float)))
    def _metrics(ranked, k, gain):
        g = ranked.astype(float) if gain == "linear" else np.power(2.0, ranked) - 1.0
        rel = np.flatnonzero(ranked > 0)
        disc = 1.0 / np.log2(np.arange(2, k + 2))
        dcg = (g[:k] * disc[: len(g[:k])]).sum()
        ideal = np.sort(g)[::-1][:k]
        idcg = (ideal * disc[: len(ideal)]).sum()
        return {f"hit@{k}": float((ranked[:k] > 0).any()), f"ndcg@{k}": float(dcg / idcg) if idcg > 0 else 0.0,
                "mrr": float(1.0 / (rel[0] + 1)) if len(rel) else 0.0, "p@1": float(ranked[0] > 0)}
    class MetricAccumulator:
        def __init__(self, k=10, gain="linear", seed=42):
            self.k, self.gain, self.rng, self.rows = k, gain, np.random.default_rng(seed), []
        def add(self, scores, labels):
            labels = np.asarray(labels)
            if not (labels > 0).any():
                return None
            m = _metrics(labels[_rank(scores, self.rng)], self.k, self.gain); self.rows.append(m); return m
        def per_slate(self):
            return {key: np.array([r[key] for r in self.rows]) for key in self.rows[0]} if self.rows else {}
    def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
        values = np.asarray(list(values), float); rng = np.random.default_rng(seed)
        means = values[rng.integers(0, len(values), size=(n_boot, len(values)))].mean(axis=1)
        return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

METRICS = ["hit@10", "ndcg@10", "mrr", "p@1"]

def evaluate(df, score):
    """Rank each slate in df (rows grouped by user) by score; one row of metrics per slate."""
    users = df.user.values
    assert (np.diff(users) >= 0).all(), "rows must be grouped by user"
    starts = np.r_[0, np.flatnonzero(np.diff(users)) + 1]; ends = np.r_[starts[1:], len(users)]
    s, y = np.asarray(score, float), df.label.values
    acc, kept = MetricAccumulator(gain=NDCG_GAIN, seed=SEED), []
    for a, b in zip(starts, ends):
        if acc.add(s[a:b], y[a:b]) is not None:
            kept.append(users[a])
    out = pd.DataFrame(acc.per_slate()); out.insert(0, "user", kept)
    return out

def summarise(per, name):
    row = {"model": name, "slates": len(per)}
    row.update({m: per[m].mean() for m in METRICS})
    for m in ["hit@10", "ndcg@10"]:
        lo, hi = bootstrap_ci(per[m]); row[f"{m} 95% CI"] = f"{lo:.3f} to {hi:.3f}"
    return row

FMT = {m: "{:.4f}" for m in METRICS} | {"slates": "{:,}"}
BASELINE_NAMES = {"score_random": "Random", "score_query": "Query only", "score_pop": "Popularity",
                  "score_repeat_pop": "Repeat → popularity", "score_mf": "MF (truncated SVD)",
                  "score_t3_maxprob": "Team 3 NCF (as published: max class prob.)",
                  "score_t3_pengaged": "Team 3 NCF (fixed: P(engaged))"}

results will upload to My Drive/BT4222_step3_lightgbm_test_08Nov_1659


In [ ]:
if IN_COLAB and DRIVE_ARTIFACT_FOLDER:
    drive_download(["features.parquet", "users.parquet", "config.json", "baseline_scores.parquet"])
cfg = json.load(open(f"{ART_DIR}/config.json"))
FEATURES = cfg["FEATURES"]
FEATURES = [c for c in FEATURES if c != "slate_len"]
CATEGORICAL = ["item_last_type", "cand_cate1"]
K = pd.read_parquet(f"{ART_DIR}/features.parquet",
                    columns=sorted(set(["user", "wid", "label", "split", "fit", "train_eval", "pop_w0", "pop_w1", "pop_w2"] + FEATURES)))
U = pd.read_parquet(f"{ART_DIR}/users.parquet")
B = pd.read_parquet(f"{ART_DIR}/baseline_scores.parquet")
BASE_COLS = [c for c in B.columns if c.startswith("score_")]
K["cand_cate1"] = pd.Categorical(K.cand_cate1)
K["item_last_type"] = K.item_last_type.astype("category")

LABEL_DERIVED = {"label", "n_pos", "split", "fit", "train_eval", "t3_list"}
assert not LABEL_DERIVED & set(FEATURES), "a label-derived column is in the feature list"
print(f"lightgbm {lgb.__version__} | {len(FEATURES)} features | step 1: MAX_USERS={cfg['MAX_USERS']}, HISTORY_CAP={cfg['HISTORY_CAP']}")
print("baselines from step 2:", [BASELINE_NAMES.get(c, c) for c in BASE_COLS])

assert cfg["MAX_USERS"] == 0 and len(U) == 171728 and len(K) == 6690773 and len(B) == 3966935, "not the full step 1 and 2 output"
assert "user_hist_len" in U.columns, "the segment cell needs user_hist_len"

downloaded features.parquet
downloaded users.parquet
downloaded config.json
downloaded baseline_scores.parquet
lightgbm 4.6.0 | 65 features | step 1: MAX_USERS=0, HISTORY_CAP=None
baselines from step 2: ['Random', 'Query only', 'Popularity', 'Repeat → popularity', 'MF (truncated SVD)', 'Team 3 NCF (as published: max class prob.)', 'Team 3 NCF (fixed: P(engaged))']


In [ ]:
# F7 (step 1): popularity for a train row no longer counts that user's own actions. Step 2's popularity and
# repeat -> popularity scores were made from the features at the time step 2 ran. Check they match the current
# features.parquet; if not, recompute both exactly as step 2 does, so train, validation and test use current values.
assert {"pop_w0", "pop_w1", "pop_w2", "item_days_since"} <= set(K.columns)
chk = B[["user", "wid"]].merge(K[["user", "wid", "split", "pop_w0", "pop_w1", "pop_w2", "item_days_since"]],
                               on=["user", "wid"], how="left", validate="one_to_one")
assert chk.split.notna().all(), "baseline_scores.parquet has rows that are not in features.parquet"
pop_now = (chk.pop_w0 + chk.pop_w1 + chk.pop_w2).to_numpy(float)
seen_now = chk.item_days_since.notna().to_numpy()
pop_old = B.score_pop.to_numpy(float)
seen_old = B.score_repeat_pop.to_numpy(float) >= 1e9                 # step 2: score = 1e9 * (in own history) + popularity
stale = ~np.isclose(pop_old, pop_now, rtol=1e-6, atol=1e-6, equal_nan=True) | (seen_old != seen_now) \
        | ~np.isclose(B.score_repeat_pop.to_numpy(float) - seen_old * 1e9, pop_now, rtol=1e-6, atol=1e-6, equal_nan=True)
split_name = chk.split.map({0: "train", 1: "val", 2: "test"}).values
print("share of rows where step 2's popularity scores differ from the current features:",
      pd.Series(stale).groupby(split_name).mean().round(4).to_dict())
if stale.any():
    B["score_pop"], B["score_repeat_pop"] = pop_now, seen_now * 1e9 + pop_now
    print(f"recomputed score_pop and score_repeat_pop from features.parquet: {stale.sum():,} rows changed "
          f"({pd.Series(stale).groupby(split_name).sum().to_dict()})")
else:
    print("step 2's popularity scores are current: no change")
del chk, pop_now, seen_now, pop_old, seen_old, stale, split_name; gc.collect()


share of rows where step 2's popularity scores differ from the current features: {'test': 0.0, 'train': 0.0075, 'val': 0.0}
recomputed score_pop and score_repeat_pop from features.parquet: 6,720 rows changed ({'test': 0, 'train': 6720, 'val': 0})


2015

## 2. Train

In [ ]:
TR = K[K.fit.values]                                         # sampled train rows, grouped by user
VA = K[K.split.values == 1].reset_index(drop=True)            # full validation slates
y_tr = (TR.label.values > 0).astype(int)
print(f"train rows {len(TR):,} (positive rate {y_tr.mean():.1%}) | validation rows {len(VA):,} in {VA.user.nunique():,} slates")

# group (slate sizes) is there only because LightGBM's NDCG stopping metric needs it; the binary objective ignores it
dtrain = lgb.Dataset(TR[FEATURES], y_tr, group=TR.groupby("user", sort=False).size().values,
                     categorical_feature=CATEGORICAL, free_raw_data=False)
dval = lgb.Dataset(VA[FEATURES], (VA.label.values > 0).astype(int), group=VA.groupby("user", sort=False).size().values,
                   categorical_feature=CATEGORICAL, reference=dtrain)
PARAMS = dict(objective="binary", metric="ndcg", eval_at=[10], learning_rate=0.05, num_leaves=63,
              min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=1.0, seed=SEED, verbose=-1)
if LOAD_MODEL_FROM:
    model = load_frozen_model("lgbm_bce.txt")      # test scores exactly the model picked on validation
else:
    t0 = time.time()
    model = lgb.train(PARAMS, dtrain, num_boost_round=3000, valid_sets=[dval], valid_names=["val"],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(100)])
    print(f"best iteration {model.best_iteration} | {time.time() - t0:.0f}s")
model.save_model(f"{OUT_DIR}/lgbm_bce.txt")

train rows 2,937,620 (positive rate 8.7%) | validation rows 1,538,177 in 17,172 slates


loaded frozen model lgbm_bce.txt from My Drive/BT4222_step3_lightgbm_val_08Nov_1257: 864 trees, no retraining


In [ ]:
import itertools

N_TRIALS, MAX_MINUTES = 15, 45
SEARCH = dict(num_leaves=[31, 63, 127], min_data_in_leaf=[100, 200, 500, 1000], feature_fraction=[0.5, 0.65, 0.8],
              bagging_fraction=[0.6, 0.8, 1.0], lambda_l2=[0.0, 1.0, 5.0, 20.0])

def tuning_row(trial, cfg, m, per, tr):
    return {"trial": trial, **cfg, "best_iter": m.best_iteration, **{f"val_{k}": per[k].mean() for k in METRICS},
            "train_hit@10": tr["hit@10"].mean(), "train_ndcg@10": tr["ndcg@10"].mean(),
            "gap_hit@10": tr["hit@10"].mean() - per["hit@10"].mean(), "gap_ndcg@10": tr["ndcg@10"].mean() - per["ndcg@10"].mean()}

if RUN_TUNING:
    assert len(FEATURES) == 65, "the feature list must stay at the locked 65"
    TE_T = K[K.train_eval.values].reset_index(drop=True)
    dtune = lgb.Dataset(TR[FEATURES], y_tr, group=TR.groupby("user", sort=False).size().values, categorical_feature=CATEGORICAL,
                        free_raw_data=False, params={"feature_pre_filter": False})
    dval_t = lgb.Dataset(VA[FEATURES], (VA.label.values > 0).astype(int), group=VA.groupby("user", sort=False).size().values,
                         categorical_feature=CATEGORICAL, reference=dtune, free_raw_data=False, params={"feature_pre_filter": False})
    per_ref = evaluate(VA, model.predict(VA[FEATURES], num_iteration=model.best_iteration))
    tr_ref = evaluate(TE_T, model.predict(TE_T[FEATURES], num_iteration=model.best_iteration))
    ref_cfg = {k: PARAMS[k] for k in SEARCH}
    rows, per_trial = [tuning_row(0, ref_cfg, model, per_ref, tr_ref)], {0: per_ref}

    grid = [dict(zip(SEARCH, v)) for v in itertools.product(*SEARCH.values())]
    grid = [g for g in grid if g != ref_cfg]
    picks = np.random.default_rng(SEED).permutation(len(grid))[:N_TRIALS]
    t_start = time.time()
    for trial, i in enumerate(picks, start=1):
        if time.time() - t_start > MAX_MINUTES * 60:
            print("time limit reached"); break
        cfg = grid[i]; t0 = time.time()
        m = lgb.train({**PARAMS, **cfg, "feature_pre_filter": False}, dtune, num_boost_round=3000, valid_sets=[dval_t], valid_names=["val"],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        per = evaluate(VA, m.predict(VA[FEATURES], num_iteration=m.best_iteration))
        tr = evaluate(TE_T, m.predict(TE_T[FEATURES], num_iteration=m.best_iteration))
        rows.append(tuning_row(trial, cfg, m, per, tr)); per_trial[trial] = per
        pd.DataFrame(rows).to_csv(f"{OUT_DIR}/tuning_trials.csv", index=False)
        print(f"trial {trial}/{len(picks)} | {cfg} | iter {m.best_iteration} | val Hit@10 {rows[-1]['val_hit@10']:.4f} "
              f"| val NDCG@10 {rows[-1]['val_ndcg@10']:.4f} | {time.time() - t0:.0f}s")
        del m; gc.collect()

    T = pd.DataFrame(rows).sort_values(["val_hit@10", "val_ndcg@10"], ascending=False).reset_index(drop=True)
    display_cols = ["trial", *SEARCH, "best_iter", "val_hit@10", "val_ndcg@10", "gap_hit@10", "gap_ndcg@10"]
    print(T[display_cols].round(4).to_string(index=False))
    top = int(T[T.trial != 0].trial.iloc[0])
    for mt in ["hit@10", "ndcg@10"]:
        diff = per_trial[top][mt] - per_ref[mt]; lo, hi = bootstrap_ci(diff)
        print(f"top trial {top} minus reference, {mt}: {diff.mean():+.4f} (95% CI {lo:+.4f} to {hi:+.4f})")
    del dtune, dval_t; gc.collect()
else:
    # The search ran once, on 4 Nov (12 of 15 trials within MAX_MINUTES), and uploaded tuning_trials.csv to TUNING_FOLDER.
    # Here: show that trial table, then retrain only the top trial and re-check its paired gain over PARAMS (~5 min).
    T = None
    if IN_COLAB:
        try:
            import io
            from googleapiclient.http import MediaIoBaseDownload
            svc = _drive()
            q = f"name='{TUNING_FOLDER}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
            for fo in svc.files().list(q=q, fields="files(id)").execute()["files"]:
                hit = svc.files().list(q=f"'{fo['id']}' in parents and name='tuning_trials.csv' and trashed=false",
                                       fields="files(id)").execute()["files"]
                if hit:
                    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, svc.files().get_media(fileId=hit[0]["id"])); done = False
                    while not done:
                        _, done = dl.next_chunk()
                    buf.seek(0); T = pd.read_csv(buf); print(f"loaded tuning_trials.csv from My Drive/{TUNING_FOLDER}")
                    break
        except Exception as e:
            print("could not read the saved trials from Drive:", e)
    if T is None:
        # copied from this notebook's saved output of the 4 Nov search (Section 2), in case the folder is not shared with you
        print(f"tuning_trials.csv not found in a folder named {TUNING_FOLDER}: using the table printed by that run")
        T = pd.DataFrame([
            (0, 63, 200, 0.80, 0.8, 1.0, 864, 0.7188, 0.4146, 0.0153, 0.0130),
            (1, 31, 200, 0.65, 0.8, 1.0, 758, 0.7176, 0.4125, 0.0054, 0.0052),
            (2, 31, 200, 0.65, 0.6, 20.0, 858, 0.7191, 0.4126, 0.0059, 0.0055),
            (3, 63, 100, 0.80, 1.0, 0.0, 445, 0.7190, 0.4125, 0.0081, 0.0078),
            (4, 31, 100, 0.50, 0.6, 0.0, 919, 0.7173, 0.4124, 0.0055, 0.0056),
            (5, 63, 100, 0.50, 0.8, 20.0, 724, 0.7190, 0.4129, 0.0111, 0.0101),
            (6, 127, 100, 0.50, 1.0, 0.0, 418, 0.7180, 0.4142, 0.0169, 0.0136),
            (7, 63, 100, 0.50, 1.0, 5.0, 590, 0.7185, 0.4135, 0.0077, 0.0097),
            (8, 63, 1000, 0.80, 0.8, 5.0, 484, 0.7187, 0.4132, 0.0082, 0.0071),
            (9, 63, 100, 0.65, 0.6, 20.0, 608, 0.7188, 0.4132, 0.0082, 0.0089),
            (10, 63, 100, 0.65, 1.0, 0.0, 570, 0.7190, 0.4129, 0.0085, 0.0090),
            (11, 127, 1000, 0.65, 0.6, 1.0, 466, 0.7209, 0.4142, 0.0126, 0.0120),
            (12, 127, 100, 0.65, 0.8, 5.0, 570, 0.7208, 0.4147, 0.0155, 0.0170)],
            columns=["trial", *SEARCH, "best_iter", "val_hit@10", "val_ndcg@10", "gap_hit@10", "gap_ndcg@10"])
    T = T.sort_values(["val_hit@10", "val_ndcg@10"], ascending=False).reset_index(drop=True)
    display_cols = ["trial", *SEARCH, "best_iter", "val_hit@10", "val_ndcg@10", "gap_hit@10", "gap_ndcg@10"]
    print("saved search (trial 0 = PARAMS):"); print(T[display_cols].round(4).to_string(index=False))

    top_row = T[T.trial != 0].iloc[0]
    cfg = {k: (int(top_row[k]) if k in ("num_leaves", "min_data_in_leaf") else float(top_row[k])) for k in SEARCH}
    dtune = lgb.Dataset(TR[FEATURES], y_tr, group=TR.groupby("user", sort=False).size().values, categorical_feature=CATEGORICAL,
                        free_raw_data=False, params={"feature_pre_filter": False})
    dval_t = lgb.Dataset(VA[FEATURES], (VA.label.values > 0).astype(int), group=VA.groupby("user", sort=False).size().values,
                         categorical_feature=CATEGORICAL, reference=dtune, free_raw_data=False, params={"feature_pre_filter": False})
    t0 = time.time()
    m = lgb.train({**PARAMS, **cfg, "feature_pre_filter": False}, dtune, num_boost_round=3000, valid_sets=[dval_t], valid_names=["val"],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    per_top = evaluate(VA, m.predict(VA[FEATURES], num_iteration=m.best_iteration))
    per_ref = evaluate(VA, model.predict(VA[FEATURES], num_iteration=model.best_iteration))
    print(f"\nre-check, top trial {int(top_row.trial)} {cfg} | iter {m.best_iteration} | val Hit@10 {per_top['hit@10'].mean():.4f} "
          f"(saved {top_row['val_hit@10']:.4f}) | {time.time() - t0:.0f}s")
    for mt in ["hit@10", "ndcg@10"]:
        diff = per_top[mt] - per_ref[mt]; lo, hi = bootstrap_ci(diff)
        print(f"top trial minus PARAMS, {mt}: {diff.mean():+.4f} (95% CI {lo:+.4f} to {hi:+.4f})"
              + ("  -> proven gain: revisit D20" if lo > 0 else ""))
    del m, dtune, dval_t; gc.collect()

loaded tuning_trials.csv from My Drive/BT4222_step3_tuning_search
saved search (trial 0 = PARAMS):
 trial  num_leaves  min_data_in_leaf  feature_fraction  bagging_fraction  lambda_l2  best_iter  val_hit@10  val_ndcg@10  gap_hit@10  gap_ndcg@10
    11         127              1000              0.65               0.6        1.0        466      0.7209       0.4142      0.0126       0.0120
    12         127               100              0.65               0.8        5.0        570      0.7208       0.4147      0.0155       0.0170
     2          31               200              0.65               0.6       20.0        858      0.7191       0.4126      0.0059       0.0055
     5          63               100              0.50               0.8       20.0        724      0.7190       0.4129      0.0111       0.0101
    10          63               100              0.65               1.0        0.0        570      0.7190       0.4129      0.0085       0.0090
     3          63             

## 3. Results

**RESULT: validation.** Every model is scored by the same code on the same validation slates. The training row (10,000 train users' full slates, D12) shows how much the model overfits.

In [ ]:
def with_baselines(split_code):
    """Our model's scores for one split, joined to step 2's baseline scores on (user, product)."""
    part = K[K.split.values == split_code].reset_index(drop=True)
    part["score_lgbm"] = model.predict(part[FEATURES], num_iteration=model.best_iteration)
    part = part.merge(B[["user", "wid"] + BASE_COLS], on=["user", "wid"], how="left", validate="one_to_one")
    assert part[BASE_COLS].notna().all().all(), "step 2 scores are missing for some rows: re-run step 2 on this step 1 output"
    return part

def results(part):
    per = {BASELINE_NAMES.get(c, c): evaluate(part, part[c]) for c in BASE_COLS}
    per["LightGBM BCE (ours)"] = evaluate(part, part.score_lgbm)
    return per, pd.DataFrame([summarise(p, n) for n, p in per.items()]).set_index("model")

PV = with_baselines(1)
per_val, res_val = results(PV)
display(res_val.style.format(FMT))
res_val.to_csv(f"{OUT_DIR}/step3_val_results.csv")

TE = K[K.train_eval.values].reset_index(drop=True)
TE["score_lgbm"] = model.predict(TE[FEATURES], num_iteration=model.best_iteration)
tr_row = summarise(evaluate(TE, TE.score_lgbm), "LightGBM BCE (ours)")
print("TRAIN (10,000 users, full slates):", {m: round(tr_row[m], 4) for m in METRICS})
print("VAL                              :", {m: round(res_val.loc["LightGBM BCE (ours)", m], 4) for m in METRICS})

# a re-run (e.g. the final RUN_TEST run) retrains the model: it must reproduce the frozen validation figures
for m, v in EXPECTED_VAL.items():
    got = res_val.loc["LightGBM BCE (ours)", m]
    print(f"val {m}: {got:.4f} (frozen {v})" + ("" if abs(got - v) < 0.0015 else "  <-- MISMATCH: do not report this run's test numbers"))

,slates,hit@10,ndcg@10,mrr,p@1,hit@10 95% CI,ndcg@10 95% CI
model,,,,,,,
Random,"17,172",0.2240,0.0826,0.1008,0.0302,0.218 to 0.230,0.080 to 0.085
Query only,"17,172",0.2349,0.0912,0.1085,0.0368,0.228 to 0.241,0.088 to 0.094
Popularity,"17,172",0.6437,0.3448,0.3388,0.1972,0.637 to 0.651,0.340 to 0.350
Repeat → popularity,"17,172",0.6813,0.3856,0.3869,0.2463,0.675 to 0.689,0.381 to 0.391
MF (truncated SVD),"17,172",0.6029,0.3196,0.3181,0.1837,0.596 to 0.610,0.315 to 0.324
Team 3 NCF (as published: max class prob.),"17,172",0.2533,0.1079,0.1214,0.0477,0.247 to 0.260,0.105 to 0.111
Team 3 NCF (fixed: P(engaged)),"17,172",0.4507,0.2187,0.2244,0.1165,0.443 to 0.458,0.214 to 0.223
LightGBM BCE (ours),"17,172",0.7188,0.4146,0.4155,0.2717,0.712 to 0.726,0.409 to 0.420


TRAIN (10,000 users, full slates): {'hit@10': np.float64(0.7341), 'ndcg@10': np.float64(0.4276), 'mrr': np.float64(0.4297), 'p@1': np.float64(0.286)}
VAL                              : {'hit@10': np.float64(0.7188), 'ndcg@10': np.float64(0.4146), 'mrr': np.float64(0.4155), 'p@1': np.float64(0.2717)}
val hit@10: 0.7188 (frozen 0.7188)
val ndcg@10: 0.4146 (frozen 0.4146)


**RESULT: train.** Every baseline and our model on the 10,000 train users' full slates (D12). For the baselines nothing is fitted on slate labels, so their train and validation scores should be close; for our model the gap shows overfitting. Popularity scores use the F7 fix (a train row does not count its own user's actions; checked in Section 1).

In [ ]:
assert (TE.split.values == 0).all(), "train_eval rows must be train users"
PTR = TE.merge(B[["user", "wid"] + BASE_COLS], on=["user", "wid"], how="left", validate="one_to_one")
assert PTR[BASE_COLS].notna().all().all(), "step 2 scores are missing for some train rows: re-run step 2 on this step 1 output"
per_tr, res_train = results(PTR)
display(res_train.style.format(FMT))
res_train.to_csv(f"{OUT_DIR}/step3_train_results.csv")

gap = pd.DataFrame({f"{m} {s}": r[m] for m in ["hit@10", "ndcg@10"] for s, r in [("train", res_train), ("val", res_val)]})
for m in ["hit@10", "ndcg@10"]:
    gap[f"{m} gap"] = gap[f"{m} train"] - gap[f"{m} val"]
print("train minus validation:")
display(gap.style.format("{:.4f}"))

def long_table(res, split):
    t = res.reset_index(); t.insert(1, "split", split); return t
SUMMARY = pd.concat([long_table(res_train, "train"), long_table(res_val, "val")], ignore_index=True)
del PTR; gc.collect()


,slates,hit@10,ndcg@10,mrr,p@1,hit@10 95% CI,ndcg@10 95% CI
model,,,,,,,
Random,"10,000",0.2264,0.0835,0.0991,0.0298,0.218 to 0.234,0.080 to 0.087
Query only,"10,000",0.2367,0.0922,0.1070,0.0364,0.229 to 0.244,0.088 to 0.096
Popularity,"10,000",0.6401,0.3405,0.3371,0.1973,0.631 to 0.650,0.334 to 0.347
Repeat → popularity,"10,000",0.6777,0.3819,0.3847,0.2444,0.669 to 0.687,0.375 to 0.389
MF (truncated SVD),"10,000",0.6014,0.3192,0.3218,0.1913,0.592 to 0.611,0.313 to 0.326
Team 3 NCF (as published: max class prob.),"10,000",0.2816,0.1269,0.1373,0.0598,0.272 to 0.290,0.122 to 0.132
Team 3 NCF (fixed: P(engaged)),"10,000",0.4998,0.2568,0.2599,0.1434,0.491 to 0.509,0.250 to 0.263
LightGBM BCE (ours),"10,000",0.7341,0.4276,0.4297,0.2860,0.725 to 0.743,0.420 to 0.435


train minus validation:


,hit@10 train,hit@10 val,ndcg@10 train,ndcg@10 val,hit@10 gap,ndcg@10 gap
model,,,,,,
Random,0.2264,0.2240,0.0835,0.0826,0.0024,0.0009
Query only,0.2367,0.2349,0.0922,0.0912,0.0018,0.0011
Popularity,0.6401,0.6437,0.3405,0.3448,-0.0036,-0.0043
Repeat → popularity,0.6777,0.6813,0.3819,0.3856,-0.0036,-0.0038
MF (truncated SVD),0.6014,0.6029,0.3192,0.3196,-0.0015,-0.0004
Team 3 NCF (as published: max class prob.),0.2816,0.2533,0.1269,0.1079,0.0283,0.0190
Team 3 NCF (fixed: P(engaged)),0.4998,0.4507,0.2568,0.2187,0.0491,0.0381
LightGBM BCE (ours),0.7341,0.7188,0.4276,0.4146,0.0153,0.0130


73

### Is the gain over the best baseline real?

For each validation user: our model's score minus the best baseline's score, bootstrapped over users. If the 95% CI excludes 0, the gain is not luck.

In [ ]:
best = res_val.drop("LightGBM BCE (ours)")["ndcg@10"].idxmax()
d = per_val["LightGBM BCE (ours)"].merge(per_val[best], on="user", suffixes=("", "_base"))
rows = []
for m in METRICS:
    diff = d[m] - d[f"{m}_base"]; lo, hi = bootstrap_ci(diff)
    rows.append({"metric": m, "ours": d[m].mean(), best: d[f"{m}_base"].mean(), "gain": diff.mean(), "95% CI": f"{lo:+.4f} to {hi:+.4f}"})
print("best baseline on validation NDCG@10:", best)
display(pd.DataFrame(rows).set_index("metric").style.format({"ours": "{:.4f}", best: "{:.4f}", "gain": "{:+.4f}"}))

best baseline on validation NDCG@10: Repeat → popularity


,ours,Repeat → popularity,gain,95% CI
metric,,,,
hit@10,0.7188,0.6813,+0.0375,+0.0324 to +0.0428
ndcg@10,0.4146,0.3856,+0.0289,+0.0262 to +0.0319
mrr,0.4155,0.3869,+0.0286,+0.0253 to +0.0320
p@1,0.2717,0.2463,+0.0254,+0.0204 to +0.0302


## 4. What the model relies on

In [ ]:
imp = pd.Series(model.feature_importance("gain"), index=FEATURES)
imp = (imp / imp.sum()).sort_values(ascending=False)
display(imp.head(15).to_frame("share of total gain").style.format("{:.1%}"))
print("features never used:", imp[imp == 0].index.tolist())

,share of total gain
pop_impr_train_slate_pct,27.1%
pop_w0_slate_pct,24.8%
item_days_since,8.1%
shop_share_w0_slate_pct,7.2%
cand_cate1,6.4%
cate3_share_w0_slate_pct,4.1%
name_overlap_slate_pct,2.6%
pop_w1,2.2%
pop_impr_train,1.8%
shop_pop,1.5%


features never used: ['cand_has_meta', 'item_flw', 'query_seen_before']


## 5. Where it wins and loses (segments)

The same validation users split into groups, comparing our model with the best baseline inside each group (rubric aspect 4). Small groups have wide CIs: only read a difference as real if its CI excludes 0.
- **History length**: does the model need history, and how does it do for light users?
- **Engaged product**: "all cold" means no engaged product in the slate was touched in any train user's history, so popularity and MF score them all 0.

In [ ]:
seg = d.copy()
hl = U.set_index("user").user_hist_len.reindex(seg.user).values
seg["history length"] = pd.cut(hl, [-1, 4, 100, 10**9], labels=["<5 actions", "5-100", "100+"])
pos = PV[PV.label.values > 0]
cold = (pos.pop_w0 + pos.pop_w1 + pos.pop_w2).values == 0
seg["engaged product"] = pd.Series(cold).groupby(pos.user.values).all().reindex(seg.user).map({True: "all cold", False: "some warm"}).values

def seg_table(col, m="ndcg@10"):
    out = []
    for g, s in seg.groupby(col, observed=True):
        diff = s[m] - s[f"{m}_base"]; lo, hi = bootstrap_ci(diff)
        out.append({col: g, "slates": len(s), "ours": s[m].mean(), best: s[f"{m}_base"].mean(),
                    "gain": diff.mean(), "95% CI": f"{lo:+.3f} to {hi:+.3f}"})
    return pd.DataFrame(out).set_index(col)

for col in ["history length", "engaged product"]:
    print(f"NDCG@10 by {col}")
    display(seg_table(col).style.format({"ours": "{:.4f}", best: "{:.4f}", "gain": "{:+.4f}"}))

NDCG@10 by history length


,slates,ours,Repeat → popularity,gain,95% CI
history length,,,,,
<5 actions,766,0.3871,0.3485,+0.0386,+0.026 to +0.051
5-100,9098,0.4083,0.3799,+0.0284,+0.025 to +0.033
100+,7308,0.4253,0.3967,+0.0286,+0.024 to +0.033


NDCG@10 by engaged product


,slates,ours,Repeat → popularity,gain,95% CI
engaged product,,,,,
all cold,3117,0.1979,0.1271,+0.0708,+0.063 to +0.078
some warm,14055,0.4627,0.4430,+0.0197,+0.017 to +0.023


## 6. Test (final run only)

Runs only when `RUN_TEST = True`, after every choice (features, hyperparameters, extensions) has been made on validation.

In [ ]:
if RUN_TEST:
    PT = with_baselines(2)
    _, res_test = results(PT)
    display(res_test.style.format(FMT))
    res_test.to_csv(f"{OUT_DIR}/step3_test_results.csv")
    # test scores in the same format as the validation file, so a blend picked on validation can be applied to test
    PT[["user", "wid", "split", "label", "score_lgbm"]].to_parquet(f"{OUT_DIR}/lgbm_scores_test.parquet", index=False)
    SUMMARY = pd.concat([SUMMARY, long_table(res_test, "test")], ignore_index=True)
else:
    print("RUN_TEST is False: test users not scored.")
PV[["user", "wid", "split", "label", "score_lgbm"]].to_parquet(f"{OUT_DIR}/lgbm_scores_val.parquet", index=False)

# one long table (model, split, metrics) for the group results table: LightGBM BCE and every baseline
SUMMARY.to_csv(f"{OUT_DIR}/step3_lightgbm_summary.csv", index=False)
print(SUMMARY.pivot(index="model", columns="split", values="hit@10").reindex(columns=["train", "val", "test"]).round(4).to_string())
print("saved", sorted(os.listdir(OUT_DIR)))


,slates,hit@10,ndcg@10,mrr,p@1,hit@10 95% CI,ndcg@10 95% CI
model,,,,,,,
Random,"17,176",0.2220,0.0812,0.0991,0.0296,0.215 to 0.228,0.078 to 0.084
Query only,"17,176",0.2388,0.0931,0.1081,0.0367,0.232 to 0.245,0.090 to 0.096
Popularity,"17,176",0.6343,0.3395,0.3362,0.1982,0.627 to 0.642,0.335 to 0.345
Repeat → popularity,"17,176",0.6762,0.3840,0.3875,0.2488,0.669 to 0.683,0.379 to 0.390
MF (truncated SVD),"17,176",0.6017,0.3191,0.3210,0.1888,0.595 to 0.609,0.314 to 0.324
Team 3 NCF (as published: max class prob.),"17,176",0.2538,0.1053,0.1189,0.0454,0.247 to 0.260,0.102 to 0.109
Team 3 NCF (fixed: P(engaged)),"17,176",0.4427,0.2134,0.2188,0.1124,0.436 to 0.449,0.209 to 0.218
LightGBM BCE (ours),"17,176",0.7179,0.4121,0.4125,0.2689,0.711 to 0.725,0.407 to 0.418


split                                        train     val    test
model                                                             
LightGBM BCE (ours)                         0.7341  0.7188  0.7179
MF (truncated SVD)                          0.6014  0.6029  0.6017
Popularity                                  0.6401  0.6437  0.6343
Query only                                  0.2367  0.2349  0.2388
Random                                      0.2264  0.2240  0.2220
Repeat → popularity                         0.6777  0.6813  0.6762
Team 3 NCF (as published: max class prob.)  0.2816  0.2533  0.2538
Team 3 NCF (fixed: P(engaged))              0.4998  0.4507  0.4427
saved ['lgbm_bce.txt', 'lgbm_scores_test.parquet', 'lgbm_scores_val.parquet', 'step3_lightgbm_summary.csv', 'step3_test_results.csv', 'step3_train_results.csv', 'step3_val_results.csv']


In [ ]:
# KEY RESULTS: our base model and the best baseline. Test rows appear only when RUN_TEST = True
KEY = ["hit@10", "ndcg@10", "mrr", "p@1"]
key_results = SUMMARY.loc[SUMMARY.model.isin(["LightGBM BCE (ours)", best]), ["model", "split", "slates"] + KEY].reset_index(drop=True)
key_results.to_csv(f"{OUT_DIR}/step3_lightgbm_key_results.csv", index=False)
print(f"KEY RESULTS (full slates; best baseline = {best}, picked on validation NDCG@10)")
print(key_results.round(4).to_string(index=False))

KEY RESULTS (full slates; best baseline = Repeat → popularity, picked on validation NDCG@10)
              model split  slates  hit@10  ndcg@10    mrr    p@1
Repeat → popularity train   10000  0.6777   0.3819 0.3847 0.2444
LightGBM BCE (ours) train   10000  0.7341   0.4276 0.4297 0.2860
Repeat → popularity   val   17172  0.6813   0.3856 0.3869 0.2463
LightGBM BCE (ours)   val   17172  0.7188   0.4146 0.4155 0.2717
Repeat → popularity  test   17176  0.6762   0.3840 0.3875 0.2488
LightGBM BCE (ours)  test   17176  0.7179   0.4121 0.4125 0.2689


In [ ]:
from googleapiclient.http import MediaFileUpload
assert RESULT_FOLDER != DRIVE_ARTIFACT_FOLDER   # set in Section 1: a new folder for every run
svc = _drive()
fid = svc.files().create(body={"name": RESULT_FOLDER, "mimeType": "application/vnd.google-apps.folder"}, fields="id").execute()["id"]
for f in sorted(os.listdir(OUT_DIR)):
    media = MediaFileUpload(f"{OUT_DIR}/{f}", resumable=True, chunksize=100 * 1024 * 1024)
    svc.files().create(body={"name": f, "parents": [fid]}, media_body=media).execute()
    print("uploaded", f)
print(f"GROUP CHAT: step3_lightgbm results in My Drive/{RESULT_FOLDER} (RUN_TEST = {RUN_TEST})")

uploaded lgbm_bce.txt
uploaded lgbm_scores_test.parquet
uploaded lgbm_scores_val.parquet
uploaded step3_lightgbm_key_results.csv
uploaded step3_lightgbm_summary.csv
uploaded step3_test_results.csv
uploaded step3_train_results.csv
uploaded step3_val_results.csv
GROUP CHAT: step3_lightgbm results in My Drive/BT4222_step3_lightgbm_test_08Nov_1659 (RUN_TEST = True)


## Findings (full data, full slates; validation unless marked Test)
- **Our base model beats every baseline.** Validation Hit@10 **0.719** (95% CI 0.712 to 0.726) vs **0.681** for repeat → popularity, the best baseline; NDCG@10 0.415 vs 0.386. The paired gain is above zero on all four metrics: Hit@10 +0.038 (CI +0.032 to +0.043), NDCG@10 +0.029 (CI +0.026 to +0.032), MRR +0.029, P@1 +0.025.
- **Against our prior baseline**, team 3's hybrid NCF retrained on our split, the gain is much larger: Hit@10 0.719 vs 0.451 (scored by P(engaged)) and 0.253 (scored as they published).
- **Mild overfitting:** training Hit@10 0.734 vs validation 0.719 (gap 0.015); NDCG@10 0.428 vs 0.415 (gap 0.013).
- **Baselines score the same on train as on validation, as they should:** none is fitted on slate labels. Repeat → popularity train Hit@10 0.678 vs validation 0.681; popularity 0.640 vs 0.644; MF 0.601 vs 0.603; random 0.226 vs 0.224. Team 3's NCF, which is trained, overfits more than our model: 0.500 vs 0.451 by P(engaged) (gap 0.049, vs our 0.015). F7 check: step 2's popularity scores were out of date for 6,720 train rows (0.75%) and were recomputed from the current features; validation and test rows were already current.
- **Tuning found no proven gain.** A random search over 12 configurations (15 planned; 45-minute limit) on validation. The top trial (127 leaves, `min_data_in_leaf` 1000, `feature_fraction` 0.65, `bagging_fraction` 0.6, `lambda_l2` 1.0) scored Hit@10 0.7209 vs 0.7188: +0.0020 (CI −0.0003 to +0.0044), with NDCG@10 −0.0004 (CI −0.0015 to +0.0009). Its train–validation gap was similar (Hit@10 0.013 vs 0.015). So the hand-set `PARAMS` stay (D20).
- **It wins in every segment, most where the baselines are blind.** When every engaged product in a slate is cold (no train user ever acted on it, so popularity and MF score it 0), NDCG@10 rises from 0.127 to 0.198 (+0.071, CI +0.063 to +0.078; 3,117 slates). Light users with fewer than 5 actions gain +0.039 (CI +0.026 to +0.051; 766 slates). Users with 5 to 100 and 100+ actions both gain about +0.029.
- **What it relies on:** the two within-slate popularity percentiles carry over half the total gain (`pop_impr_train_slate_pct` 27.1%, `pop_w0_slate_pct` 24.8%), followed by days since the user last touched this exact product (`item_days_since` 8.1%), the within-slate percentile of shop affinity (7.2%) and the level-1 category (6.4%). So the model mainly asks "how popular is this product compared with the others in this slate", then "has this user touched it, or its shop, before". Percentiles are computed on full slates for every split (step 1 fix), and `slate_len` was removed, since it is the same for every product in a slate. Three features are never used: `cand_has_meta`, `item_flw`, `query_seen_before`.
- **Reproducible:** re-running on 8 Nov gave exactly the same model (best iteration 864, Hit@10 0.7188). The test run loads that saved model (`LOAD_MODEL_FROM = BT4222_step3_lightgbm_val_08Nov_1257`).
- **Test (17,176 users, scored once on 8 Nov with the saved model):** Hit@10 **0.7179** vs **0.6762** for repeat → popularity; NDCG@10 0.4121 vs 0.3840. Close to validation (0.7188 and 0.4146), so choosing settings on validation did not inflate the scores much. Train–test gap: Hit@10 0.016, NDCG@10 0.015. The baselines keep their validation order (test Hit@10: popularity 0.634, MF 0.602, team 3 NCF 0.443 by P(engaged) and 0.254 as published, query only 0.239, random 0.222). Paired test CIs are in `step4_blend_and_segments.ipynb`.
- **Paired test CI (step 4, 17,176 users):** the gain over repeat → popularity is proven, Hit@10 +0.0416 (95% CI +0.0369 to +0.0468) and NDCG@10 +0.0281 (CI +0.0256 to +0.0306).

**Next:** LightGBM LambdaRank (`step3_lambdarank.ipynb`) was picked as the final model on validation (8 Nov). This notebook stays as the BCE reference it is compared against.